# Python pour la Data Engineering

Notebook de l'article [Comment utiliser Python pour la Data Engineering ?](https://moncoachdata.com/blog/python-pour-la-data-engineering/) de MonCoachData.

Tout est simulé en local : aucun serveur ni compte cloud n'est nécessaire. PySpark demande Java (déjà installé sur Google Colab). Airflow n'est pas inclus ici : c'est un service à installer à part, voir l'article.

## Installation

Sur Google Colab, `moto`, `great-expectations` et `pyarrow` ne sont pas installés par défaut.

In [ ]:
%pip install -q "moto[s3]" great-expectations pyarrow

## Versions utilisées

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # PySpark prévient qu'il ne gère pas encore tout de pandas 3

import numpy as np
import pandas as pd
import requests
import sqlalchemy
import boto3
import dask
import pyspark
import great_expectations as gx

print("Python", sys.version.split()[0])
for module in (pd, np, requests, sqlalchemy, boto3, dask, pyspark, gx):
    print(f"{module.__name__:<20}{module.__version__}")

## Données de démonstration

1 000 ventes avec des valeurs manquantes et des villes écrites de plusieurs façons.

In [ ]:
rng = np.random.default_rng(42)   # générateur aléatoire reproductible
n = 1_000

ventes = pd.DataFrame({
    "id": np.arange(1, n + 1),
    "category": rng.choice(["A", "B", "C"], size=n),
    "sub_category": rng.choice(["x", "y"], size=n),
    "value": rng.integers(10, 500, size=n).astype(float),
    "ville": rng.choice(["PARIS", "LYON", "Genève", "zurich"], size=n),
})

# on abîme volontairement quelques lignes, comme dans la vraie vie
ventes.loc[rng.choice(n, 30, replace=False), "value"] = np.nan
ventes.loc[rng.choice(n, 20, replace=False), "ville"] = None

print(ventes.shape)
ventes.head()

## #2 Extraction

### API

Une petite API locale expose les ventes (uniquement pour la démonstration).

In [ ]:
import http.server
import threading

class Handler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        corps = ventes.to_json(orient="records").encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(corps)

    def log_message(self, *args):   # pas de log dans la sortie
        pass

serveur = http.server.HTTPServer(("127.0.0.1", 8000), Handler)
threading.Thread(target=serveur.serve_forever, daemon=True).start()
print("API de démonstration prête sur http://127.0.0.1:8000/data")

Extraction avec `requests`.

In [ ]:
import requests

response = requests.get("http://127.0.0.1:8000/data", timeout=10)
if response.status_code == 200:
    data = response.json()

print(response.status_code, response.headers["Content-Type"])
print(len(data), "enregistrements")
print(data[0])

Liste de dictionnaires → DataFrame.

In [ ]:
df_api = pd.DataFrame(data)

print(df_api.shape)
df_api.dtypes

### Base de données

SQLite à la place de PostgreSQL : seule l'URL de connexion change.

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///entrepot.db")

# on remplit d'abord une base « source » pour l'exemple
ventes.to_sql("ventes", con=engine, if_exists="replace", index=False)

df = pd.read_sql("SELECT * FROM ventes", con=engine)
print(df.shape)
df.head(3)

Laisse la base faire les agrégations.

In [ ]:
requete = """
SELECT category,
       COUNT(*)              AS nb_ventes,
       ROUND(AVG(value), 1)  AS moyenne
FROM ventes
WHERE value IS NOT NULL
GROUP BY category
ORDER BY category
"""
pd.read_sql(requete, con=engine)

### Stockage cloud

S3 simulé avec `moto` : le code `boto3` est celui de la production.

In [ ]:
import os
import boto3
from moto import mock_aws

# Clés factices + S3 simulé par moto : aucun compte AWS n'est nécessaire pour tester
os.environ.update(AWS_ACCESS_KEY_ID="test", AWS_SECRET_ACCESS_KEY="test", AWS_DEFAULT_REGION="eu-west-3")
mock = mock_aws()
mock.start()

s3 = boto3.client("s3")
s3.create_bucket(Bucket="bucket_name", CreateBucketConfiguration={"LocationConstraint": "eu-west-3"})

# on dépose un fichier dans le bucket pour l'exemple
ventes.to_csv("data.csv", index=False)
s3.upload_file("data.csv", "bucket_name", "data.csv")

s3.download_file("bucket_name", "data.csv", "local_data.csv")
print(pd.read_csv("local_data.csv").shape)

## #3 Transformation

### Nettoyer avec pandas

In [ ]:
df = pd.read_csv("local_data.csv")
print("Valeurs manquantes avant :")
print(df.isna().sum()[lambda s: s > 0].to_string())

df = df.dropna()
df["ville"] = df["ville"].str.lower()

print("\nLignes restantes :", len(df))
print(df["ville"].value_counts().to_string())

### Agréger

In [ ]:
df_grouped = df.groupby("category")["value"].sum()
print(df_grouped)

Tableau croisé.

In [ ]:
df_pivot = df.pivot_table(index="category", columns="sub_category", values="value", aggfunc="mean")
df_pivot.round(1)

**Piège :** `groupby().sum()` sans choisir de colonne colle les textes bout à bout.

In [ ]:
piege = df.groupby("category").sum()      # sans sélectionner de colonne
print(piege.columns.tolist())
print(piege.loc["A", "sub_category"][:25], "…")   # les textes sont collés bout à bout

### Dask pour les gros fichiers

In [ ]:
import dask.dataframe as dd

# un fichier plus gros (500 000 lignes) pour l'exemple
grand = pd.DataFrame({
    "category": rng.choice(["A", "B", "C"], size=500_000),
    "value": rng.integers(10, 500, size=500_000),
})
grand.to_csv("large_data.csv", index=False)

df_large = dd.read_csv("large_data.csv", blocksize="1MB")
print("Nombre de partitions :", df_large.npartitions)

df_transformed = df_large.groupby("category")["value"].sum().compute()
print(df_transformed)

Même résultat que pandas.

In [ ]:
attendu = grand.groupby("category")["value"].sum()
print("Même résultat que pandas :", df_transformed.sort_index().equals(attendu))

## #4 Chargement

### Base de données

In [ ]:
df.to_sql("ventes_propres", con=engine, if_exists="replace", index=False)

pd.read_sql("SELECT COUNT(*) AS nb_lignes FROM ventes_propres", con=engine)

### Fichiers : CSV et Parquet

In [ ]:
df.to_csv("clean_data.csv", index=False)
df.to_parquet("clean_data.parquet", index=False)

for fichier in ("clean_data.csv", "clean_data.parquet"):
    print(f"{fichier:<22}{os.path.getsize(fichier):>8} octets")

### Stockage cloud

In [ ]:
s3.upload_file("clean_data.csv", "bucket_name", "clean_data.csv")
s3.upload_file("clean_data.parquet", "bucket_name", "clean_data.parquet")

for objet in s3.list_objects_v2(Bucket="bucket_name")["Contents"]:
    print(f"{objet['Key']:<22}{objet['Size']:>8} octets")

## #6 PySpark (mode local)

PySpark demande que Java soit installé.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[2]")                      # 2 cœurs de ma machine, pas de cluster
         .appName("DataEngineering")
         .config("spark.ui.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

df_spark = spark.read.csv("large_data.csv", header=True, inferSchema=True)
print("Lignes lues :", df_spark.count())

df_spark = df_spark.dropna().groupBy("category").sum("value")
df_spark.orderBy("category").show()

## #7 Qualité des données avec Great Expectations

Syntaxe de la version 1.x.

In [ ]:
import great_expectations as gx
import great_expectations.expectations as gxe

context = gx.get_context(mode="ephemeral")
context.variables.progress_bars = {"globally": False, "metric_calculations": False}   # sortie plus lisible
batch = context.data_sources.pandas_default.read_dataframe(df)

attente = gxe.ExpectColumnValuesToBeInSet(column="category", value_set=["A", "B", "C"])
resultat = batch.validate(attente)
print("Validation réussie :", resultat.success)

Avec des catégories inattendues, la validation échoue.

In [ ]:
df_abime = pd.concat([df, pd.DataFrame({"category": ["D", "D"], "sub_category": ["x", "y"], "value": [50.0, 70.0]})])
batch_abime = context.data_sources.pandas_default.read_dataframe(df_abime)

resultat = batch_abime.validate(attente)
print("Validation réussie :", resultat.success)
print("Valeurs inattendues :", resultat.result["unexpected_count"], "sur", resultat.result["element_count"])
print("Exemples :", resultat.result["partial_unexpected_list"])

## #8 Bonnes pratiques : un mini-pipeline

In [ ]:
import logging

logger = logging.getLogger("pipeline")
logger.setLevel(logging.INFO)
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(handler)

def extraire(url):
    response = requests.get(url, timeout=5)
    response.raise_for_status()          # lève une erreur si le statut n'est pas 2xx
    return pd.DataFrame(response.json())

def transformer(df):
    df = df.dropna().copy()
    df["ville"] = df["ville"].str.lower()
    return df

def charger(df, table, engine):
    df.to_sql(table, con=engine, if_exists="replace", index=False)
    return len(df)

def pipeline(url):
    try:
        brut = extraire(url)
        propre = transformer(brut)
        n = charger(propre, "ventes_pipeline", engine)
        logger.info("%d lignes lues, %d lignes chargées", len(brut), n)
        return True
    except requests.exceptions.RequestException as erreur:
        logger.error("Extraction impossible (%s)", type(erreur).__name__)
        return False

pipeline("http://127.0.0.1:8000/data")

Si l'API est injoignable, l'erreur est interceptée et journalisée.

In [ ]:
pipeline("http://127.0.0.1:9999/data")   # aucun serveur sur ce port